# 25.2 — Inductive logic programming

**Lesson tagline.** Inductive logic programming searches for a rule whose symbolic coverage explains the examples and rejects the counterexamples.

ILP learns readable relational clauses from positive and negative ground atoms. Examples prune a symbolic hypothesis space rather than merely fitting weights.

Save a copy to Drive to edit.

In [ ]:

import math
import itertools
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

SEED = 25
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))


def binary_cross_entropy(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0 - 1e-12)
    y = np.asarray(y, dtype=float)
    terms = -y * np.log(p) - (1.0 - y) * np.log(1.0 - p)
    return float(np.mean(terms))



def parent2(facts, x, z):
    parents = facts.get('parent', set())
    for y in {b for a, b in parents if a == x}:
        if (y, z) in parents:
            return True
    return False


def covers(rule, facts, pair):
    x, z = pair
    if rule == 'parent(X,Z)':
        return pair in facts.get('parent', set())
    if rule == 'parent2(X,Z)':
        return parent2(facts, x, z)
    if rule == 'anything(X,Z)':
        return x != z
    if rule == 'same_address(X,Z)':
        return pair in facts.get('same_address', set())
    if rule == 'coworker_parent2(X,Z)':
        return parent2(facts, x, z) or pair in facts.get('coworker', set())
    raise ValueError(rule)


def score_rule(rule, facts, positives, negatives):
    cov_pos = sum(covers(rule, facts, pair) for pair in positives)
    cov_neg = sum(covers(rule, facts, pair) for pair in negatives)
    return cov_pos - cov_neg, cov_pos, cov_neg


def enumerate_rules(include_distractors=True):
    rules = ['parent(X,Z)', 'parent2(X,Z)', 'anything(X,Z)']
    if include_distractors:
        rules.extend(['same_address(X,Z)', 'coworker_parent2(X,Z)'])
    return rules


def make_ilp_ladder():
    d1_facts = {'parent': {('ann', 'bob'), ('bob', 'cara'), ('cara', 'dan')}}
    rungs = []
    rungs.append({'name': 'D1 lesson family chain', 'facts': d1_facts, 'positives': {('ann', 'cara'), ('bob', 'dan')}, 'negatives': {('ann', 'dan'), ('bob', 'cara'), ('cara', 'ann')}, 'rules': enumerate_rules(False)})
    names = ['ann', 'bob', 'cara', 'dan', 'eli', 'fay']
    parents = {(names[i], names[i + 1]) for i in range(len(names) - 1)}
    positives = {(names[i], names[i + 2]) for i in range(len(names) - 2)}
    negatives = {('ann', 'dan'), ('bob', 'eli'), ('fay', 'ann')}
    rungs.append({'name': 'D2 longer clean kinship chain', 'facts': {'parent': parents}, 'positives': positives, 'negatives': negatives, 'rules': enumerate_rules(True)})
    facts3 = {'parent': parents, 'same_address': {('ann', 'cara'), ('bob', 'dan'), ('ann', 'eli')}, 'coworker': {('cara', 'eli'), ('dan', 'fay')}}
    rungs.append({'name': 'D3 negatives and distracting predicates', 'facts': facts3, 'positives': positives, 'negatives': negatives | {('ann', 'eli'), ('cara', 'eli')}, 'rules': enumerate_rules(True)})
    facts4 = {'parent': parents | {('gina', 'hen'), ('hen', 'ira')}, 'same_address': {('ann', 'cara'), ('gina', 'ira')}, 'coworker': {('ann', 'gina'), ('bob', 'hen')}}
    positives4 = positives | {('gina', 'ira')}
    negatives4 = negatives | {('ann', 'gina'), ('gina', 'ann'), ('hen', 'gina')}
    rungs.append({'name': 'D4 inline family-company graph', 'facts': facts4, 'positives': positives4, 'negatives': negatives4, 'rules': enumerate_rules(True)})
    long_names = [f'p{i}' for i in range(11)]
    parents5 = {(long_names[i], long_names[i + 1]) for i in range(len(long_names) - 1)}
    parents5.remove(('p4', 'p5'))
    positives5 = {(long_names[i], long_names[i + 2]) for i in range(len(long_names) - 2) if i != 3}
    negatives5 = {(long_names[i], long_names[j]) for i in range(0, 8, 2) for j in range(4, 11, 3) if j != i + 2}
    facts5 = {'parent': parents5, 'same_address': set(positives5) | {('p0', 'p9'), ('p2', 'p10')}, 'coworker': {('p1', 'p8'), ('p3', 'p7'), ('p5', 'p9')}}
    rungs.append({'name': 'D5 larger missing-fact overbroad stress test', 'facts': facts5, 'positives': positives5, 'negatives': negatives5, 'rules': enumerate_rules(True)})
    return rungs


## The concept, built once (D1)

ILP scores clauses by coverage: $score(r)=cov^+(r)-cov^-(r)$ where $cov^+(r)=|\{e\in P:r\models e\}|$ and $cov^-(r)=|\{e\in N:r\models e\}|$.

In [ ]:

facts = {'parent': {('ann', 'bob'), ('bob', 'cara'), ('cara', 'dan')}}
positives = {('ann', 'cara'), ('bob', 'dan')}
negatives = {('ann', 'dan'), ('bob', 'cara'), ('cara', 'ann')}
for rule in ['parent(X,Z)', 'parent2(X,Z)', 'anything(X,Z)']:
    print(rule, score_rule(rule, facts, positives, negatives))
assert score_rule('parent(X,Z)', facts, positives, negatives)[0] == -1
assert score_rule('parent2(X,Z)', facts, positives, negatives)[0] == 2
assert score_rule('anything(X,Z)', facts, positives, negatives)[0] == -1


The two-hop rule proves both positives and no negatives, while the one-hop and overbroad clauses fail for different reasons.

In [ ]:

before = parent2(facts, 'ann', 'dan')
facts_added = {'parent': set(facts['parent']) | {('bob', 'dan')}}
after = parent2(facts_added, 'ann', 'dan')
print('grandparent(ann,dan) before:', int(before))
print('grandparent(ann,dan) after:', int(after))
assert int(before) == 0
assert int(after) == 1
assert score_rule('parent2(X,Z)', facts, positives, negatives)[1:] == (2, 0)


## The dataset ladder

The F16 ladder increases graph size, distracting predicates, negative examples, and missing facts.

In [ ]:

rungs = make_ilp_ladder()
for i, rung in enumerate(rungs, start=1):
    fact_count = sum(len(v) for v in rung['facts'].values())
    print(f"D{i}: {rung['name']} facts={fact_count} positives={len(rung['positives'])} negatives={len(rung['negatives'])} rules={len(rung['rules'])}")
    print('sample positives:', list(sorted(rung['positives']))[:3])


## Run the SAME method across D1–D5

In [ ]:

results = []
for i, rung in enumerate(rungs, start=1):
    scored = []
    for rule in rung['rules']:
        score, cov_pos, cov_neg = score_rule(rule, rung['facts'], rung['positives'], rung['negatives'])
        scored.append((score, cov_pos, cov_neg, rule))
    best = max(scored)
    solved = int(best[1] == len(rung['positives']) and best[2] == 0)
    results.append({'D': i, 'best_rule': best[3], 'score': best[0], 'solved': solved, 'searched': len(scored), 'scored': scored})
print('rung | best_rule | score | solved | candidates')
for row in results:
    print(f"D{row['D']} | {row['best_rule']} | {row['score']} | {row['solved']} | {row['searched']}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, row in enumerate(results):
    scored = sorted(row['scored'], reverse=True)
    labels = [item[3].replace('(X,Z)', '') for item in scored]
    values = [item[0] for item in scored]
    axes[0, col].barh(labels, values)
    axes[0, col].set_title(f"D{col + 1} coverage")
    axes[0, col].set_xlabel('score')
axes[1, 0].plot([row['D'] for row in results], [row['score'] for row in results], marker='o', label='best score')
axes[1, 0].plot([row['D'] for row in results], [row['searched'] for row in results], marker='s', label='candidates')
axes[1, 0].set_title('score and search vs complexity')
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Pitfall on the hardest rung

On D5, positives-only scoring rewards lazy overbroad clauses. Adding negative coverage and a small language bias recovers the two-hop rule.

In [ ]:

d5 = rungs[-1]
positives_only = []
for rule in d5['rules']:
    score, cov_pos, cov_neg = score_rule(rule, d5['facts'], d5['positives'], set())
    positives_only.append((cov_pos, rule))
print('positives-only winner:', max(positives_only))
regularized = []
for rule in d5['rules']:
    score, cov_pos, cov_neg = score_rule(rule, d5['facts'], d5['positives'], d5['negatives'])
    length_penalty = 0.1 if rule == 'coworker_parent2(X,Z)' else 0.0
    regularized.append((score - length_penalty, rule, cov_pos, cov_neg))
print('negative-aware winner:', max(regularized))


## Evaluate it + Practice

- **Metric.** Track solved tasks, best rule score, and candidates searched, and compare it with a positives-only coverage rule.
- **Cheap sanity check.** D1 scores must be -1, 2, and -1 for parent, parent2, and anything.
- **Ablation.** remove negative examples and watch anything(X,Z) become competitive.
- **Failure signals.** best score rises while false positive coverage also rises.

### Practice prompts


- Add a sibling predicate and define a new candidate rule.

- Increase maximum clause length and count candidates searched.

- Construct negatives that distinguish same-address from grandparent.